# organic_ratio — entry notebook


## 1. Git sync

Подтянуть актуальный `main` из `Anton-Filimoncev-azur/organic_ratio` поверх локальной копии. Требует `GIT_PAT` в `.env`.

In [1]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "Anton-Filimoncev-azur"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio
HEAD is now at dab9add  work
HEAD is now at dab9add  work
Repository synced successfully.


From https://github.com/Anton-Filimoncev-azur/organic_ratio
 * branch            main       -> FETCH_HEAD


## 2. Env

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [3]:
# %run run.py

## 4. Per-source preprocessing (raw → data/features/partition/)

In [3]:
# import gc
# %run run_preprocessing.py
# gc.collect()

## 5. Cohort aggregation

`run_cohort_aggregation.py` — мерджит per-source user-grain фичи и роллапит до cohort-grain по ключам из `cohort.keys`. Добавляет `cohort_size` и `n_calendar_days`.

In [ ]:
%run run_cohort_aggregation.py

## 6. Target + features build (model-ready train / test)

`run_target_build.py` за один проход:
1. Считает target (`organic_share`, `total_installs`, `organic_installs`) на грануляции `cohort.keys − media_source`.
2. Реагрегирует все user-grain фичи на ту же грануляцию (политика SUM/MEAN из `aggregate_to_cohort`).
3. Джойнит target + features, пишет:
   - `data/features/targets/targets.parquet` (полный + колонка `split`)
   - `data/train/targets_train.parquet` (split == train, без `split`)
   - `data/test/targets_test.parquet` (split == test, без `split`)

После этого `targets_train.parquet` / `targets_test.parquet` — это уже готовые для модели таблицы.

In [3]:
import gc
%run run_target_build.py
gc.collect()

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
Cohort keys: ['platform', 'media_source', 'country_code', 'campaign', 'install_date']
Target keys (media_source dropped): ['platform', 'country_code', 'campaign', 'install_date']
Reading installs: data/features/partition/installs.parquet
Target table shape: (545519, 8)
shape: (10, 8)
┌──────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────┐
│ platform ┆ country_cod ┆ campaign   ┆ install_da ┆ total_inst ┆ organic_in ┆ organic_sh ┆ split  │
│ ---      ┆ e           ┆ ---        ┆ te         ┆ alls       ┆ stalls     ┆ are        ┆ ---    │
│ str      ┆ ---         ┆ str        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ str    │
│          ┆ str         ┆            ┆ datetime[m ┆ u32        ┆ u32        ┆ f64        ┆        │
│          ┆             ┆            ┆ s]         ┆            ┆            ┆            ┆        │
╞══════════╪═════════════╪════════════╪════════════╪═══

1567

## 6. Train / Eval / Inference (TODO)

PyMC иерархическая Beta-регрессия для cohort-level organic share.

In [ ]:
# %run run_train.py
# %run run_eval.py
# %run run_inference.py

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [ ]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)